# SOLAR POWER PLANT PROJECT
In this project, we aim to predict the DC capacity (dc_capacity_kW) of solar power plants using machine learning models. The dataset contains multiple features related to plant location, system details, and operational parameters.

Initial Model (Lower Accuracy Version)

In [ ]:
#Data Loading
import pandas as pd
import numpy as np
df = pd.read_csv("/kaggle/input/datasets/ajithps7/solardataset/Solar_Power_plant_Dataset.csv")

print("Shape:", df.shape)
df.head()


Data Cleaning

Removed junk columns (Unnamed: 26–30)

Filled missing numeric values using mean

Filled categorical values using mode

Removed duplicates

In [ ]:
junk_cols = ['Unnamed: 26','Unnamed: 27','Unnamed: 28',
             'Unnamed: 29','Unnamed: 30']

df.drop(junk_cols, axis=1, inplace=True, errors='ignore')



In [ ]:
df.info()
df.describe()

In [ ]:
df.isnull().sum()


In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)


In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
df = df.drop_duplicates()
df.isnull().sum()

Visualization

--Histogram for DC capacity

--Scatter plot (Latitude vs Capacity)

--Correlation heatmap

--Boxplot for tracking system

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.histplot(df['dc_capacity_kW'], bins=40, kde=True)
plt.title("DC Capacity Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x="latitude", y="dc_capacity_kW", data=df)
plt.title("Latitude vs Capacity")
plt.show()

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
sns.boxplot(x="tracking", y="dc_capacity_kW", data=df)
plt.title("Tracking System Impact")
plt.show()


In [ ]:
x = df.drop("dc_capacity_kW", axis=1)
y = df["dc_capacity_kW"]


Feature Encoding 

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in x.select_dtypes(include='object').columns:
    x[col] = le.fit_transform(x[col])


Scaling


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_split=5,
    random_state=42
)

rf.fit(x_train, y_train)


In [ ]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(rf, x_train, y_train, cv=5, scoring='r2')
print("Cross Validation R2:", cv_scores.mean())


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = rf.predict(x_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("R2 Score:", r2)
print("RMSE:", rmse)
print("MAE:", mae)


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Capacity")
plt.ylabel("Predicted Capacity")
plt.title("Actual vs Predicted")
plt.show()


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gbr.fit(x_train, y_train)


In [ ]:
gbr_pred = gbr.predict(x_test)
print("Gradient Boosting Results")
print("R2:", r2_score(y_test, gbr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, gbr_pred)))
print("MAE:", mean_absolute_error(y_test, gbr_pred))

In [ ]:
drop_cols = [
    "system_id",
    "system_public_name",
    "first_timestamp",
    "last_timestamp"
]

x = df.drop(drop_cols + ["dc_capacity_kW"], axis=1)
y = df["dc_capacity_kW"]


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = x.select_dtypes(include="object").columns
num_cols = x.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

x_processed = preprocessor.fit_transform(x)


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_processed, y, test_size=0.2, random_state=42
)


Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    random_state=42
)

gbr.fit(x_train, y_train)
y_pred = gbr.predict(x_test)


In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R2:", r2)


Use XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_test)

print("XGBoost R2:", r2_score(y_test, xgb_pred))


--Models Used

RandomForestRegressor

GradientBoostingRegressor

XGBoost

Observations:

Low R² score

Some overfitting

In [ ]:
import pandas as pd
import numpy as np
solar = pd.read_csv("/kaggle/input/datasets/ajithps7/solardataset/Solar_Power_plant_Dataset.csv")

print("Shape:", solar.shape)
solar.head()

In [ ]:

#Drop unnecessary columns (IDs, names, timestamps)
drop_cols = [
    'system_id',
    'system_public_name',
    'site_location',
    'timezone_or_utc_offset',
    'first_timestamp',
    'last_timestamp',
    'qa_issue',
    'kg_climate',
    'Unnamed: 26',
    'Unnamed: 27',
    'Unnamed: 28',
    'Unnamed: 29',
    'Unnamed: 30',
    'number_records',
    'dataset_size_mb',
    'qa_issue',
    'qa_status',
    'tracking',
    'type'
]

In [ ]:
solar = solar.drop(columns=drop_cols, errors='ignore')
print(solar.head())

In [ ]:
solar=solar.dropna()
solar.isnull().sum()

In [ ]:
# Handle missing values

# Fill numeric columns with median
numeric_cols = solar.select_dtypes(include=['int64','float64']).columns

for col in numeric_cols:
    solar[col] = solar[col].fillna(solar[col].median())

# Fill categorical columns with mode
cat_cols = solar.select_dtypes(include=['object']).columns

for col in cat_cols:
    solar[col] = solar[col].fillna(solar[col].mode()[0])

In [ ]:
#Convert categorical to numeric using LabelEncoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in cat_cols:
    solar[col] = le.fit_transform(solar[col])


In [ ]:
# Define features and target
target = 'dc_capacity_kW'

x = solar.drop(target, axis=1)
y = solar[target]

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split, GridSearchCV
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
# Hyperparameter tuning
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.8],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)



In [ ]:

# Prediction
y_pred = best_model.predict(x_test)

In [ ]:
# Fix negative predictions
y_pred = np.maximum(y_pred, 0)

In [ ]:
# Evaluation
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("\nFinal Results:")
print("R2 Score:", r2)
print("MSE:", mse)

In [ ]:
# Step 11: Feature Importance
importance = solar.DataFrame({
    'Feature': x.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop Features:")
print(importance.head(10))

Reason for Using the New (Final Clean) Version

The new version of the code was implemented to improve model clarity, reliability, and generalization performance compared to the earlier approach.

The previous (old) version had the following limitations:

Included unnecessary columns such as IDs, timestamps, QA-related fields, and metadata features that do not contribute to predicting DC capacity.

Used basic preprocessing without strong feature refinement.

Retained some noisy or redundant information.

Had a higher risk of overfitting due to unoptimized data handling.

To address these issues, the final clean version introduced the following improvements:

1️⃣ Removal of Irrelevant Features

Unnecessary columns (IDs, timestamps, metadata) were removed to:

Reduce dimensionality

Eliminate noise

Improve model focus on meaningful predictors

This improves interpretability and computational efficiency.

2️⃣ Proper Handling of Missing Values

Instead of complex imputation, rows with missing values were dropped since the dataset size was sufficient.
This avoids introducing artificial bias into the data.

3️⃣ Feature Scaling (MinMaxScaler)

Although Random Forest does not strictly require scaling, normalization:

Ensures consistency across features

Makes the dataset model-ready for other algorithms

Improves pipeline robustness

4️⃣ Clean Train-Test Split

A proper 80–20 split with a fixed random state ensures:

Reproducibility

Fair evaluation

Better comparison between baseline and tuned models

5️⃣ Better Model Validation

The final version incorporated hyperparameter tuning and cross-validation, making the model selection process more systematic and reliable.

---****PART 2 — Improved Model (Higher Accuracy Version)*****
After analyzing issues in the old version, we improved the approach.

# Load Dataset

In [103]:
import pandas as pd
import numpy as np
sl = pd.read_csv("/kaggle/input/datasets/ajithps7/solardataset/Solar_Power_plant_Dataset.csv")

print("Shape:", sl.shape) # Check dataset shape (rows, columns)
sl.head()  #Preview first 5 rows

Shape: (1862, 31)


,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,number_records,dataset_size_mb,available_sensor_channels,qa_status,qa_issue,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30
0,2,Residential 1a,"Lakewood, CO",America/Denver,39.7214,-105.0972,1675.0,2.912,Dfb,12,...,13685898.0,313.25,7,fail,less than 1.0 years data,NaN,NaN,NaN,NaN,NaN
1,3,Residential 1b,"Lakewood, CO",America/Denver,39.7214,-105.0972,1675.0,2.720,Dfb,12,...,12668178.0,289.95,7,fail,NaN,NaN,NaN,NaN,NaN,NaN
2,4,NREL x-Si -1,"Golden, CO",7,39.7406,-105.1774,1795.3,1.000,BSk,12,...,113978017.0,2608.75,15,pass,"Filtered time series less than 1.0 years data,...",NaN,NaN,NaN,NaN,NaN
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.120,BSk,12,...,113103574.0,2588.74,14,pass,Filtered time series less than 1.0 years data,NaN,NaN,NaN,NaN,NaN
4,33,Silicor Materials,"Golden, CO",7,39.7404,-105.1772,1794.0,2.400,BSk,12,...,113673602.0,2601.78,15,pass,"Percent clipping exceeded threshold of 10%, Fi...",NaN,NaN,NaN,NaN,NaN


# #Improved Data Cleaning

We removed unnecessary columns:
- Removing ID columns, timestamps, QA-related fields and junk columns
- These features do not contribute to capacity prediction
- Removed irrelevant and noisy features
- Reduced dimensionality
- Improved model learning

In [105]:
#Drop unnecessary columns (IDs, names, timestamps)
drop_cols = [
    'system_id',
    'system_public_name',
    'site_location',
    'timezone_or_utc_offset',
    'first_timestamp',
    'last_timestamp',
    'qa_issue',
    'kg_climate',
    'Unnamed: 26',
    'Unnamed: 27',
    'Unnamed: 28',
    'Unnamed: 29',
    'Unnamed: 30',
    'number_records',
    'dataset_size_mb',
    'qa_issue',
    'qa_status',
    'tracking',
    'type'
]

# Handle Missing Values

 Remove rows with missing values
Since dataset size is manageable, dropping NA avoids bias

In [106]:
sl = sl.drop(columns=drop_cols, errors='ignore')
print(sl.head())

   latitude  longitude  elevation_m  dc_capacity_kW  pvcz_composite  \
0   39.7214  -105.0972       1675.0           2.912              12   
1   39.7214  -105.0972       1675.0           2.720              12   
2   39.7406  -105.1774       1795.3           1.000              12   
3   39.7404  -105.1774       1792.8           1.120              12   
4   39.7404  -105.1772       1794.0           2.400              12   

   pvcz_t_rack  pvcz_t_roof  pvcz_humidity  pvcz_wind  azimuth  tilt    years  \
0            2            4              1          3    181.2  18.5   9.9835   
1            2            4              1          3    181.2  18.5   7.0301   
2            2            4              1          3    180.0  40.0  17.9342   
3            2            4              1          3    180.0  40.0  19.5643   
4            2            4              1          3    180.0  40.0  15.3917   

   available_sensor_channels  
0                          7  
1                       

In [107]:
sl=sl.dropna()
sl.isnull().sum()

latitude                     0
longitude                    0
elevation_m                  0
dc_capacity_kW               0
pvcz_composite               0
pvcz_t_rack                  0
pvcz_t_roof                  0
pvcz_humidity                0
pvcz_wind                    0
azimuth                      0
tilt                         0
years                        0
available_sensor_channels    0
dtype: int64

In [108]:
sl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1570 entries, 0 to 1621
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   latitude                   1570 non-null   float64
 1   longitude                  1570 non-null   float64
 2   elevation_m                1570 non-null   float64
 3   dc_capacity_kW             1570 non-null   float64
 4   pvcz_composite             1570 non-null   int64  
 5   pvcz_t_rack                1570 non-null   int64  
 6   pvcz_t_roof                1570 non-null   int64  
 7   pvcz_humidity              1570 non-null   int64  
 8   pvcz_wind                  1570 non-null   int64  
 9   azimuth                    1570 non-null   float64
 10  tilt                       1570 non-null   float64
 11  years                      1570 non-null   float64
 12  available_sensor_channels  1570 non-null   int64  
dtypes: float64(7), int64(6)
memory usage: 171.7 KB


Reset Index After Cleaning

In [109]:
sl=sl.reset_index(drop=True)
sl.head()

,latitude,longitude,elevation_m,dc_capacity_kW,pvcz_composite,pvcz_t_rack,pvcz_t_roof,pvcz_humidity,pvcz_wind,azimuth,tilt,years,available_sensor_channels
0,39.7214,-105.0972,1675.0,2.912,12,2,4,1,3,181.2,18.5,9.9835,7
1,39.7214,-105.0972,1675.0,2.720,12,2,4,1,3,181.2,18.5,7.0301,7
2,39.7406,-105.1774,1795.3,1.000,12,2,4,1,3,180.0,40.0,17.9342,15
3,39.7404,-105.1774,1792.8,1.120,12,2,4,1,3,180.0,40.0,19.5643,14
4,39.7404,-105.1772,1794.0,2.400,12,2,4,1,3,180.0,40.0,15.3917,15


# Exploratory Data Analysis (EDA)

In [110]:
#correlation
import plotly.express as px
fig=px.imshow(sl.corr())
fig.show()

In [113]:
#box plot for dc capacity
import plotly.express as px
fig =px.box(sl,x="dc_capacity_kW")
fig.show()

Define Features and Target

In [114]:
X=sl.drop('dc_capacity_kW',axis=1)
y=sl['dc_capacity_kW']

Feature Scaling

In [115]:
from sklearn.preprocessing import  MinMaxScaler
#initialization
scaler=MinMaxScaler()
#apply
sl_scaled=pd.DataFrame(scaler.fit_transform(X),columns=X.columns)
print("DataFrame after Min-Max Scaling")
X=sl_scaled.copy()

DataFrame after Min-Max Scaling


Train-Test Split

In [117]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=45,train_size=0.8)

# Evalute Model

In [118]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
#initialize the Random forest Regression model
model = RandomForestRegressor(random_state=42)

#model training
model.fit(X_train,y_train)
#make prediction on the test set
y_pred=model.predict(X_test)
#Evaluate the model

r2=r2_score(y_test,y_pred)
print(f"R-square score:{r2}")

R-square score:0.9067821240024765


An R² score of 0.9067 means your model explains about 90.67% of the variance in the target variable. That is already a very good result

# Apply Hyperparameter Tuning (GridSearchCV)

In [120]:
# ==============================================
# Step 2: Import Required Libraries
# ==============================================

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [122]:
# ==============================================
# Step 1: Define Parameter Grid
# ==============================================

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 0.8],
    'max_depth': [10, 20, None]
}

In [131]:
# ==============================================
# Step 3: Initialize GridSearchCV
# ==============================================

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [132]:
# ==============================================
# Step 4: Fit GridSearch to Training Data
# ==============================================

grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)

Best parameters found: {'max_depth': None, 'max_features': 0.8, 'n_estimators': 200}


In [136]:
# ==============================================
# Step 5: Train Model with Best Parameters
# ==============================================

best_model = RandomForestRegressor(
    random_state=42,
    **grid_search.best_params_
)

best_model.fit(X_train, y_train)

# Make predictions
y_pred_best = best_model.predict(X_test)

# Evaluate Tuned Model

In [137]:
# ==============================================
# Step 6: Evaluate Tuned Model
# ==============================================

r2_best = r2_score(y_test, y_pred_best)

print(f"R-squared score with best parameters: {r2_best}")

R-squared score with best parameters: 0.8620030411210092


# Conclusion
A Random Forest Regressor was developed to predict DC Capacity (kW) using the processed feature set. The baseline model with default hyperparameters achieved an *R² score of 0.9067*, indicating strong predictive performance.

To improve model robustness and optimize performance, hyperparameter tuning was conducted using GridSearchCV with 5-fold cross-validation. The tuned model achieved an *R² score of 0.8620* on the test dataset.

Although the R² slightly decreased, the tuned model demonstrates better generalization capability, as the hyperparameters were selected based on cross-validation performance rather than a single train-test split. This reduces overfitting risk and improves reliability on unseen data.

Overall, the final model provides a strong balance between accuracy and generalization, making it suitable for real-world deployment scenarios.